## 1ter. DirectML (Intel Iris Xe)

In [1]:
# Installe torch-directml uniquement si nécessaire (Windows + pas de CUDA)
import os

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import sys
import json
import time
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
if not torch.cuda.is_available():
    try:
        import torch_directml
        print("torch-directml déjà installé.")
    except Exception:
        import sys
        !{sys.executable} -m pip install torch-directml -q
        import torch_directml
        print("torch-directml installé.")

# Re-sélection du device après installation éventuelle
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    DEVICE_BACKEND = "cuda"
else:
    try:
        import torch_directml
        DEVICE = torch_directml.device()
        DEVICE_BACKEND = "directml"
    except Exception:
        DEVICE = torch.device("cpu")
        DEVICE_BACKEND = "cpu"

print("Gerät final :", DEVICE)
print("Backend final :", DEVICE_BACKEND)

ERROR: Could not find a version that satisfies the requirement torch-directml (from versions: none)
ERROR: No matching distribution found for torch-directml


ModuleNotFoundError: No module named 'torch_directml'

## 1bis. Konfiguration von Weights & Biases

In [2]:
import sys
!{sys.executable} -m pip install wandb -q
import wandb

WANDB_ENTITY = "eidl-thm"
WANDB_PROJECT = "Uebungabgabe3_Vades_Ngounouo__Lister_Melaguie"
WANDB_REQUIRED = True

# Force online sync so runs are visible to the team dashboard.
wandb.require("core")
WANDB_AVAILABLE = False
try:
    wandb.login(relogin=True)
    api = wandb.Api()
    _ = api.project(f"{WANDB_ENTITY}/{WANDB_PROJECT}")
    WANDB_AVAILABLE = True
    print(f"W&B OK: Zugriff auf {WANDB_ENTITY}/{WANDB_PROJECT}")
except Exception as e:
    WANDB_AVAILABLE = False
    print(f"W&B nicht bereit: {e}")
    if WANDB_REQUIRED:
        raise RuntimeError(
            f"W&B ist erforderlich, aber Login/Zugriff auf {WANDB_ENTITY}/{WANDB_PROJECT} fehlt. "
            "Bitte mit einem Team-Account einloggen, der Projektzugriff hat."
        ) from e

wandb: WARNING `wandb.require('core')` is a no-op as it is now the default behavior.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

  2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

  ········


wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\Lenovo\_netrc
wandb: Currently logged in as: vadesdjoko (eidl-thm) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B OK: Zugriff auf eidl-thm/Uebungabgabe3_Vades_Ngounouo__Lister_Melaguie


## 2. Laden von train.json / validation.json

In [3]:
PATH_TRAIN_JSON = "./train.json"
PATH_VAL_JSON = "./validation.json"

with open(PATH_TRAIN_JSON, "r", encoding="utf-8") as f:
    train_raw = json.load(f)

with open(PATH_VAL_JSON, "r", encoding="utf-8") as f:
    val_raw = json.load(f)

print("Typ von train_raw :", type(train_raw))
print("Anzahl der Elemente (Training) :", len(train_raw))

# Anzeige des ersten Elements, um seine genaue Struktur zu sehen
print("\nErstes Element :")
print(train_raw[0])


Typ von train_raw : <class 'list'>
Anzahl der Elemente (Training) : 214122

Erstes Element :
{'control code': 'S:2\nB:5\nE:5\nB:6\n', 'abc notation': 'X:1\nL:1/8\nM:4/4\nK:Emin\n|: E2 EF E2 EF | DEFG AFDF | E2 EF E2 B2 |1 efe^d e2 e2 :|2 efe^d e3 B |: e2 ef g2 fe | \n defg afdf |1 e2 ef g2 fe | efe^d e3 B :|2 g2 bg f2 af | efe^d e2 e2 ||'}


In [4]:
def extract_abc_field(entry):
    """
    Extrahiert das Feld mit der ABC-Notation aus einem Element des Datensatzes,
    unabhängig von Groß-/Kleinschreibung oder genauem Schlüsselnamen (z.B. 'abc notation', 'abc', 'ABC').
    Ist das Element bereits eine Zeichenkette, wird sie unverändert zurückgegeben.
    """
    if isinstance(entry, str):
        return entry

    if isinstance(entry, dict):
        for key in entry.keys():
            if "abc" in key.lower():
                return entry[key]
        raise KeyError(f"Kein Schlüssel mit 'abc' gefunden unter : {list(entry.keys())}")

    raise TypeError(f"Unerwarteter Elementtyp : {type(entry)}")


# Test am ersten Element
sample_abc = extract_abc_field(train_raw[0])
print("Erkannter ABC-Ausschnitt :\n", sample_abc[:300])


Erkannter ABC-Ausschnitt :
 X:1
L:1/8
M:4/4
K:Emin
|: E2 EF E2 EF | DEFG AFDF | E2 EF E2 B2 |1 efe^d e2 e2 :|2 efe^d e3 B |: e2 ef g2 fe | 
 defg afdf |1 e2 ef g2 fe | efe^d e3 B :|2 g2 bg f2 af | efe^d e2 e2 ||


In [5]:
train_tunes = [extract_abc_field(entry) for entry in train_raw]
val_tunes = [extract_abc_field(entry) for entry in val_raw]

print(f"Trainingsstücke : {len(train_tunes)}")
print(f"Validierungsstücke : {len(val_tunes)}")


Trainingsstücke : 214122
Validierungsstücke : 2162


## 3. Aufbau des Vokabulars (zeichenweise Tokenisierung)

In [9]:
def build_vocab(tunes):
    """Baut ein zeichenweises Vokabular aus allen Musikstücken auf."""
    all_text = "".join(tunes)
    chars = sorted(set(all_text))

    char2idx = {ch: i for i, ch in enumerate(chars)}
    idx2char = {i: ch for i, ch in enumerate(chars)}

    return char2idx, idx2char

char2idx, idx2char = build_vocab(train_tunes)
VOCAB_SIZE = len(char2idx)
print(f"Vokabulargröße : {VOCAB_SIZE}")

def encode(text, char2idx):
    """Wandelt eine Zeichenkette in eine Liste von Indizes um (ignoriert unbekannte Zeichen)."""
    return [char2idx[ch] for ch in text if ch in char2idx]


Vokabulargröße : 95


## 4. PyTorch-Dataset

In [10]:
class ABCDataset(Dataset):
    def __init__(self, tunes, char2idx, seq_length=100, stride=25):
        self.seq_length = seq_length
        self.char2idx = char2idx

        # Jedes Stück wird nur einmal kodiert (keine Duplizierung pro Fenster)
        self.encoded_tunes = [encode(tune, char2idx) for tune in tunes]

        # stride > 1 : es wird nicht bei JEDEM Zeichen ein neues Fenster erzeugt,
        # sondern nur alle `stride` Zeichen. Das reduziert die Anzahl der Beispiele
        # (und damit die Batches pro Epoche) drastisch, ohne viel Lernsignal zu verlieren,
        # da aufeinanderfolgende Fenster ohnehin fast identisch sind.
        self.index = []
        for tune_idx, encoded in enumerate(self.encoded_tunes):
            n_positions = len(encoded) - seq_length
            if n_positions > 0:
                for i in range(0, n_positions, stride):
                    self.index.append((tune_idx, i))

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        tune_idx, i = self.index[idx]
        encoded = self.encoded_tunes[tune_idx]
        seq_length = self.seq_length

        input_seq = encoded[i:i + seq_length]
        target_seq = encoded[i + 1:i + seq_length + 1]

        return torch.tensor(input_seq, dtype=torch.long), torch.tensor(target_seq, dtype=torch.long)


In [11]:
SEQ_LENGTH = 100
STRIDE = 100 

train_dataset = ABCDataset(train_tunes, char2idx, seq_length=SEQ_LENGTH, stride=STRIDE)
val_dataset = ABCDataset(val_tunes, char2idx, seq_length=SEQ_LENGTH, stride=STRIDE)

print(f"Anzahl der Trainingsbeispiele : {len(train_dataset)}")
print(f"Anzahl der Validierungsbeispiele : {len(val_dataset)}")

BATCH_SIZE = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

x_batch, y_batch = next(iter(train_loader))
print("Form input :", x_batch.shape)
print("Form target :", y_batch.shape)


Anzahl der Trainingsbeispiele : 917889
Anzahl der Validierungsbeispiele : 9103
Form input : torch.Size([64, 100])
Form target : torch.Size([64, 100])


## 5. RNN-/LSTM-Modell

In [12]:
class MusicRNN(nn.Module):
    """
    Modell zur Erzeugung von Sequenzen (ABC-Zeichen) basierend auf RNN oder LSTM.
    Nimmt eine Token-Sequenz als Eingabe und sagt das nächste Token voraus.
    """
    def __init__(self, vocab_size, embedding_dim=256, hidden_dim=512,
                 num_layers=2, rnn_type="lstm", dropout=0.3):
        super(MusicRNN, self).__init__()

        self.vocab_size = vocab_size
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.rnn_type = rnn_type.lower()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        if self.rnn_type == "lstm":
            self.rnn = nn.LSTM(
                input_size=embedding_dim,
                hidden_size=hidden_dim,
                num_layers=num_layers,
                batch_first=True,
                dropout=dropout if num_layers > 1 else 0.0
            )
        elif self.rnn_type == "rnn":
            self.rnn = nn.RNN(
                input_size=embedding_dim,
                hidden_size=hidden_dim,
                num_layers=num_layers,
                batch_first=True,
                nonlinearity="tanh",
                dropout=dropout if num_layers > 1 else 0.0
            )
        else:
            raise ValueError("rnn_type muss 'rnn' oder 'lstm' sein")

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden=None):
        batch_size = x.size(0)

        if hidden is None:
            hidden = self.init_hidden(batch_size, x.device)

        embedded = self.embedding(x)
        out, hidden = self.rnn(embedded, hidden)
        out = self.dropout(out)
        logits = self.fc(out)

        return logits, hidden

    def init_hidden(self, batch_size, device):
        if self.rnn_type == "lstm":
            h0 = torch.zeros(self.num_layers, batch_size, self.hidden_dim, device=device)
            c0 = torch.zeros(self.num_layers, batch_size, self.hidden_dim, device=device)
            return (h0, c0)
        else:
            h0 = torch.zeros(self.num_layers, batch_size, self.hidden_dim, device=device)
            return h0


In [13]:
model = MusicRNN(
    vocab_size=VOCAB_SIZE,
    embedding_dim=256,
    hidden_dim=512,
    num_layers=2,
    rnn_type="lstm"
).to(DEVICE)

print(model)

batch_size_test, seq_len_test = 4, 50
dummy_input = torch.randint(0, VOCAB_SIZE, (batch_size_test, seq_len_test)).to(DEVICE)
logits, hidden = model(dummy_input)
print("Form der Logits :", logits.shape)


MusicRNN(
  (embedding): Embedding(95, 256)
  (rnn): LSTM(256, 512, num_layers=2, batch_first=True, dropout=0.3)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=512, out_features=95, bias=True)
)
Form der Logits : torch.Size([4, 50, 95])


## 6. Training

In [ ]:
def train_one_epoch(model, dataloader, optimizer, criterion, device, clip_grad=5.0,
                     log_every=50, log_file=None, use_wandb=False, epoch=None):
    model.train()
    total_loss = 0.0
    n_batches = len(dataloader)

    for batch_idx, (inputs, targets) in enumerate(dataloader):
        try:
            inputs, targets = inputs.to(device), targets.to(device)

            optimizer.zero_grad()
            logits, _ = model(inputs)
            loss = criterion(logits.view(-1, logits.size(-1)), targets.view(-1))
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
            optimizer.step()

            total_loss += loss.item()

        except Exception as e:
            err_text = str(e).lower()
            is_cuda_oom = isinstance(e, torch.cuda.OutOfMemoryError)
            is_dml_oom = ("out of memory" in err_text) and (DEVICE_BACKEND == "directml")

            if is_cuda_oom or is_dml_oom:
                msg = f"[OOM] Batch {batch_idx}/{n_batches} - {e}"
                print(msg)
                if log_file:
                    with open(log_file, "a") as f:
                        f.write(msg + "\n")
                if getattr(device, "type", "") == "cuda":
                    torch.cuda.empty_cache()
                raise

            raise

        if batch_idx % log_every == 0:
            mem_str = ""
            gpu_mem_alloc = gpu_mem_reserved = None
            if getattr(device, "type", "") == "cuda":
                gpu_mem_alloc = torch.cuda.memory_allocated(device) / 1024**2
                gpu_mem_reserved = torch.cuda.memory_reserved(device) / 1024**2
                mem_str = f" | GPU-Speicher alloc={gpu_mem_alloc:.0f}MB reserved={gpu_mem_reserved:.0f}MB"

            msg = f"  Batch {batch_idx}/{n_batches} | loss={loss.item():.4f}{mem_str}"
            print(msg, flush=True)
            if log_file:
                with open(log_file, "a") as f:
                    f.write(msg + "\n")

            if use_wandb and wandb.run is not None:
                log_dict = {
                    "batch_loss": loss.item(),
                    "batch": batch_idx,
                }
                if epoch is not None:
                    log_dict["epoch"] = epoch
                if gpu_mem_alloc is not None:
                    log_dict["gpu_mem_alloc_MB"] = gpu_mem_alloc
                    log_dict["gpu_mem_reserved_MB"] = gpu_mem_reserved
                wandb.log(log_dict)

    return total_loss / len(dataloader)


@torch.no_grad()
def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0

    for inputs, targets in dataloader:
        inputs, targets = inputs.to(device), targets.to(device)
        logits, _ = model(inputs)
        loss = criterion(logits.view(-1, logits.size(-1)), targets.view(-1))
        total_loss += loss.item()

    return total_loss / len(dataloader)

In [ ]:
def train_model(model, train_loader, val_loader, device,
                 num_epochs=20, learning_rate=1e-3, weight_decay=0.0,
                 patience=3, save_path="best_model.pt", log_file="training_log.txt",
                 use_wandb=False, wandb_project="abc-music-rnn", wandb_run_name=None,
                 wandb_config=None, wandb_entity=None):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=1)

    history = {"train_loss": [], "val_loss": []}
    best_val_loss = float("inf")
    epochs_no_improve = 0

    if log_file:
        with open(log_file, "w") as f:
            f.write(f"Trainingsbeginn - Gerät={device}\n")

    wandb_enabled = bool(use_wandb)
    wandb_watch_enabled = False

    if wandb_enabled:
        config = dict(wandb_config) if wandb_config else {}
        config.update({
            "num_epochs": num_epochs,
            "learning_rate": learning_rate,
            "weight_decay": weight_decay,
            "batch_size": train_loader.batch_size,
            "device": str(device),
        })
        try:
            wandb.init(
                project=wandb_project,
                entity=wandb_entity,
                name=wandb_run_name,
                config=config,
                reinit=True,
            )
            wandb.watch(model, log="all", log_freq=100)
            wandb_watch_enabled = True
            print("W&B aktiviert.")
        except Exception as e:
            warn_msg = f"W&B deaktiviert (Init-Fehler): {e}"
            print(warn_msg)
            if log_file:
                with open(log_file, "a") as f:
                    f.write(warn_msg + "\n")
            wandb_enabled = False

    try:
        for epoch in range(1, num_epochs + 1):
            start = time.time()

            msg = f"--- Epoche {epoch}/{num_epochs} ---"
            print(msg)
            if log_file:
                with open(log_file, "a") as f:
                    f.write(msg + "\n")

            train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device,
                                          log_file=log_file, use_wandb=wandb_enabled, epoch=epoch)
            val_loss = evaluate(model, val_loader, criterion, device)

            scheduler.step(val_loss)

            history["train_loss"].append(train_loss)
            history["val_loss"].append(val_loss)

            elapsed = time.time() - start
            current_lr = optimizer.param_groups[0]['lr']
            msg = (f"Epoche {epoch}/{num_epochs} | "
                   f"Trainingsverlust : {train_loss:.4f} | Validierungsverlust : {val_loss:.4f} | "
                   f"LR : {current_lr:.6f} | Zeit : {elapsed:.1f}s")
            print(msg)
            if log_file:
                with open(log_file, "a") as f:
                    f.write(msg + "\n")

            if wandb_enabled and wandb.run is not None:
                wandb.log({
                    "epoch": epoch,
                    "train_loss": train_loss,
                    "val_loss": val_loss,
                    "learning_rate": current_lr,
                    "epoch_time_s": elapsed,
                })

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                epochs_no_improve = 0
                torch.save(model.state_dict(), save_path)
                if wandb_enabled and wandb.run is not None:
                    wandb.save(save_path)
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= patience:
                    msg = f"Early Stopping bei Epoche {epoch} (keine Verbesserung seit {patience} Epochen)"
                    print(msg)
                    if log_file:
                        with open(log_file, "a") as f:
                            f.write(msg + "\n")
                    break
    finally:
        # Remove forward hooks installed by wandb.watch before inference/generation.
        if wandb_watch_enabled:
            try:
                wandb.unwatch(model)
            except Exception:
                pass

        if wandb_enabled and wandb.run is not None:
            wandb.finish()

    return history

In [ ]:
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
NUM_EPOCHS = 30

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

model = MusicRNN(vocab_size=VOCAB_SIZE, embedding_dim=256, hidden_dim=512,
                  num_layers=2, rnn_type="lstm").to(DEVICE)

history = train_model(
    model, train_loader, val_loader, DEVICE,
    num_epochs=NUM_EPOCHS, learning_rate=LEARNING_RATE,
    use_wandb=WANDB_AVAILABLE,
    wandb_entity=WANDB_ENTITY,
    wandb_project=WANDB_PROJECT,
    wandb_config={
        "embedding_dim": 256,
        "hidden_dim": 512,
        "num_layers": 2,
        "rnn_type": "lstm",
        "seq_length": SEQ_LENGTH,
        "vocab_size": VOCAB_SIZE,
    },
)


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


W&B aktiviert.
--- Epoche 1/30 ---
  Batch 0/14343 | loss=4.5616
  Batch 50/14343 | loss=2.3937
  Batch 100/14343 | loss=1.8041
  Batch 150/14343 | loss=1.6299
  Batch 200/14343 | loss=1.5426
  Batch 250/14343 | loss=1.4771
  Batch 300/14343 | loss=1.3933
  Batch 350/14343 | loss=1.3920
  Batch 400/14343 | loss=1.4394
  Batch 450/14343 | loss=1.3494
  Batch 500/14343 | loss=1.2911
  Batch 550/14343 | loss=1.3300
  Batch 600/14343 | loss=1.2847
  Batch 650/14343 | loss=1.2793
  Batch 700/14343 | loss=1.2153
  Batch 750/14343 | loss=1.2253
  Batch 800/14343 | loss=1.2001
  Batch 850/14343 | loss=1.2194
  Batch 900/14343 | loss=1.2565
  Batch 950/14343 | loss=1.2496
  Batch 1000/14343 | loss=1.2042
  Batch 1050/14343 | loss=1.2087
  Batch 1100/14343 | loss=1.1634
  Batch 1150/14343 | loss=1.2038
  Batch 1200/14343 | loss=1.1516
  Batch 1250/14343 | loss=1.1664
  Batch 1300/14343 | loss=1.1806
  Batch 1350/14343 | loss=1.1429
  Batch 1400/14343 | loss=1.1362
  Batch 1450/14343 | loss=1.199

In [ ]:
plt.plot(history["train_loss"], label="Trainingsverlust")
plt.plot(history["val_loss"], label="Validierungsverlust")
plt.xlabel("Epoche")
plt.ylabel("Verlust")
plt.legend()
plt.title("Lernkurve")
plt.show()


## 7. Music Generation
After training, use the RNN to generate new tunes from a seed sequence of ABC notation.
Sampling with temperature and optional top-k filtering helps produce coherent sequences.

In [ ]:
@torch.no_grad()
def sample_next_token(logits, temperature=1.0, top_k=None):
    """Sample one token index from logits using temperature and optional top-k filtering."""
    if temperature <= 0:
        raise ValueError("temperature muss > 0 sein")

    logits = logits / temperature

    if top_k is not None and top_k > 0:
        top_k = min(top_k, logits.size(-1))
        values, indices = torch.topk(logits, top_k)
        probs = torch.softmax(values, dim=-1)
        sampled_local = torch.multinomial(probs, num_samples=1)
        next_idx = indices.gather(-1, sampled_local)
        return next_idx.item()

    probs = torch.softmax(logits, dim=-1)
    next_idx = torch.multinomial(probs, num_samples=1)
    return next_idx.item()


@torch.no_grad()
def generate_tune(model, seed_text, char2idx, idx2char, max_new_tokens=400,
                  temperature=0.9, top_k=20, device=DEVICE):
    """Generate an ABC continuation from a seed text."""
    model.eval()

    seed_ids = [char2idx[ch] for ch in seed_text if ch in char2idx]
    if len(seed_ids) == 0:
        raise ValueError("Die Seed-Sequenz enthält keine Zeichen aus dem Vokabular.")

    hidden = None

    # Warm up the hidden state with the full seed.
    seed_tensor = torch.tensor(seed_ids, dtype=torch.long, device=device).unsqueeze(0)
    logits, hidden = model(seed_tensor, hidden)

    generated_ids = list(seed_ids)

    current_token = seed_tensor[:, -1:]
    for _ in range(max_new_tokens):
        logits, hidden = model(current_token, hidden)
        next_logits = logits[:, -1, :].squeeze(0)
        next_id = sample_next_token(next_logits, temperature=temperature, top_k=top_k)

        generated_ids.append(next_id)
        current_token = torch.tensor([[next_id]], dtype=torch.long, device=device)

    return "".join(idx2char[i] for i in generated_ids)


seed_example = val_tunes[0][:200] if len(val_tunes) > 0 else train_tunes[0][:200]
generated_abc = generate_tune(
    model=model,
    seed_text=seed_example,
    char2idx=char2idx,
    idx2char=idx2char,
    max_new_tokens=500,
    temperature=0.9,
    top_k=20,
    device=DEVICE,
)

print("Seed (Ausschnitt):")
print(seed_example[:300])
print("\nGenerierte ABC-Sequenz (Ausschnitt):")
print(generated_abc[:1200])

## 8. Evaluation
Evaluate generated-music quality with next-note prediction metrics on the test split:
- Top-1 Accuracy
- Top-5 Accuracy

In [ ]:
PATH_TEST_JSON = "./test.json"

if os.path.exists(PATH_TEST_JSON):
    with open(PATH_TEST_JSON, "r", encoding="utf-8") as f:
        test_raw = json.load(f)
    test_tunes = [extract_abc_field(entry) for entry in test_raw]
    print(f"Test-Set geladen aus {PATH_TEST_JSON} | Stücke: {len(test_tunes)}")
else:
    # Fallback: use validation data as evaluation split when test.json is unavailable.
    test_tunes = val_tunes
    print("Kein test.json gefunden -> Evaluation auf validation.json (Fallback).")


TEST_STRIDE = STRIDE
test_dataset = ABCDataset(test_tunes, char2idx, seq_length=SEQ_LENGTH, stride=TEST_STRIDE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


@torch.no_grad()
def evaluate_topk(model, dataloader, device, topk=(1, 5)):
    model.eval()

    correct = {k: 0 for k in topk}
    total = 0

    for inputs, targets in dataloader:
        inputs = inputs.to(device)
        targets = targets.to(device)

        logits, _ = model(inputs)
        # logits: [B, T, V], targets: [B, T]
        bsz, seq_len, vocab_size = logits.shape

        logits_flat = logits.view(-1, vocab_size)
        targets_flat = targets.view(-1)
        total += targets_flat.size(0)

        max_k = max(topk)
        topk_indices = torch.topk(logits_flat, k=max_k, dim=-1).indices

        for k in topk:
            hits = (topk_indices[:, :k] == targets_flat.unsqueeze(1)).any(dim=1)
            correct[k] += hits.sum().item()

    metrics = {f"top_{k}_acc": correct[k] / total if total > 0 else 0.0 for k in topk}
    metrics["num_next_note_predictions"] = total
    return metrics


metrics = evaluate_topk(model, test_loader, DEVICE, topk=(1, 5))
print(f"Top-1 Accuracy: {metrics['top_1_acc']:.4f} ({metrics['top_1_acc']*100:.2f}%)")
print(f"Top-5 Accuracy: {metrics['top_5_acc']:.4f} ({metrics['top_5_acc']*100:.2f}%)")
print(f"Anzahl ausgewerteter Next-Note-Vorhersagen: {metrics['num_next_note_predictions']}")